# Investment Research Agent

Project Goal: Build a three-person agent that researches an equity (one ticker in focus) and produces a grounded analyst note covering price, recent news and earnings if present, rather than a bare BUY/SELL score.

Make the work re-runnable and auditable by a teammate.
Add enduring memory, so a second run doesn't start from zero.
The market-specialist role covers the yfinance tools, the shared ToolResult schema, safe handling of missing fields, tests without live APIs, and feeding numbers into memory.


Steps: Run the cells **top to bottom**. Steps 3-11 write each module; Step 12 runs tests; Steps 13-16 run the Investment Research agent.

| Requirement | Where |
|---|---|
| Plans research steps | `research_agent.py` `plan()` |
| Uses tools dynamically | tool registry, retry, re-plan (`_call`, `_followups`) |
| Self-reflects | `reflect()` + evaluator scores |
| Learns across runs | `memory.py` lessons -> planner + first draft |
| Prompt chaining | `chain.py` |
| Routing | `router.py` |
| Evaluator-Optimizer | `evaluator.py` |

## Step 1 - Install the dependencies

In [ ]:
!pip install -q yfinance pandas google-genai

## Step 2 - API keys (only GEMINI_API_KEY is needed for LLM mode; the rest are optional)
Get a free Gemini key at https://aistudio.google.com/apikey . Alternative: Colab **Secrets** (key icon, left sidebar) -> add GEMINI_API_KEY

In [ ]:
import os
from getpass import getpass


os.environ['GEMINI_API_KEY'] = getpass('GEMINI_API_KEY (Enter to skip): ').strip()
os.environ['NEWSAPI_KEY'] = getpass('NEWSAPI_KEY (Enter to skip): ').strip()
os.environ['FRED_API_KEY'] = getpass('FRED_API_KEY (Enter to skip): ').strip()

os.environ['SEC_USER_AGENT'] = input('SEC_USER_AGENT (Enter to skip): ').strip()
keys = ['GEMINI_API_KEY','NEWSAPI_KEY','FRED_API_KEY','SEC_USER_AGENT']
for k in keys:
    if not os.environ[k]: del os.environ[k]
print({k: bool(os.environ.get(k)) for k in keys})

GEMINI_API_KEY (Enter to skip): ··········
NEWSAPI_KEY (Enter to skip): ··········
FRED_API_KEY (Enter to skip): ··········
SEC_USER_AGENT (Enter to skip): deepikastha@gmail.com
{'GEMINI_API_KEY': True, 'NEWSAPI_KEY': True, 'FRED_API_KEY': True, 'SEC_USER_AGENT': True}


Step 3 - Shared ToolResult schema: This code is not actually fetching data or running an AI model. It is creating the standard container/contract that your tools will use to report their results.

For Investment agent architecture, we can think of schema.py as the common language between the tools and the agent.

In [ ]:
%%writefile schema.py
"""Shared ToolResult schema: every tool in the agent returns this shape."""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


@dataclass
class ToolResult:
    tool: str
    ok: bool
    data: dict[str, Any] = field(default_factory=dict)
    error: str | None = None
    warnings: list[str] = field(default_factory=list)
    source: str = ""
    fetched_at: str = field(default_factory=_now)

    @classmethod
    def success(cls, tool, data, source="", warnings=None) -> "ToolResult":
        return cls(tool=tool, ok=True, data=data, source=source, warnings=list(warnings or []))

    @classmethod
    def failure(cls, tool, error, source="", warnings=None) -> "ToolResult":
        return cls(tool=tool, ok=False, error=str(error), source=source, warnings=list(warnings or []))

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


Writing schema.py


## Step 4 - Market tools (Yahoo Finance): This file creates the market/stock analysis tools for the agent. Its main job is to get stock price history and company fundamentals from Yahoo Finance through yfinance, clean the results, calculate useful metrics, and return everything in the standardized ToolResult format from your previous schema.py. So the market tools don't just return random dictionaries but a standardized ToolResult objects. This code retrieves and calculates financial information; it does not itself tell the user what stock to buy or sell.

In [ ]:
%%writefile market_tools.py
"""Market specialist tools (yfinance). Never raise: always return a ToolResult.

`ticker_factory` lets tests inject a fake object with .history() and .info,
so no live API is needed.
"""
from __future__ import annotations

import math
from typing import Any, Callable

from schema import ToolResult

SOURCE = "Yahoo Finance (yfinance)"
PRICE_COLS = ["Open", "High", "Low", "Close", "Volume"]

NUMERIC_FIELDS = {
    "market_cap": "marketCap",
    "trailing_pe": "trailingPE",
    "forward_pe": "forwardPE",
    "trailing_eps": "trailingEps",
    "forward_eps": "forwardEps",
    "dividend_yield": "dividendYield",
    "beta": "beta",
    "revenue_growth": "revenueGrowth",
    "profit_margin": "profitMargins",
    "fifty_two_week_high": "fiftyTwoWeekHigh",
    "fifty_two_week_low": "fiftyTwoWeekLow",
}
TEXT_FIELDS = {"name": "shortName", "sector": "sector", "industry": "industry", "currency": "currency"}


def _default_factory(symbol: str):
    import yfinance as yf  # lazy import so tests don't need the package

    return yf.Ticker(symbol)


def _num(value: Any) -> float | None:
    """Coerce to a finite float, or None (handles None, NaN, 'N/A', inf)."""
    try:
        f = float(value)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(f) or math.isinf(f) else f


def _round(v: float | None, n: int = 4) -> float | None:
    return None if v is None else round(v, n)


def get_price_history(ticker: str, period: str = "6mo", interval: str = "1d",
                      ticker_factory: Callable | None = None) -> ToolResult:
    tool, sym = "get_price_history", ticker.upper().strip()
    try:
        df = (ticker_factory or _default_factory)(sym).history(period=period, interval=interval)
    except Exception as exc:  # network, bad ticker, yfinance internals
        return ToolResult.failure(tool, f"{type(exc).__name__}: {exc}", SOURCE)
    if df is None or len(df) == 0:
        return ToolResult.failure(tool, f"No price rows for '{sym}' (bad ticker or no data)", SOURCE)

    warnings: list[str] = []
    df = df.copy()
    for col in PRICE_COLS:
        if col not in df.columns:
            warnings.append(f"Column {col} missing from provider response")
            df[col] = float("nan")
    df = df.dropna(subset=["Close"])
    if df.empty:
        return ToolResult.failure(tool, "All Close values were missing", SOURCE, warnings)
    if len(df) < 20:
        warnings.append(f"Only {len(df)} rows; moving averages/volatility may be unavailable")

    close = df["Close"].astype(float)
    n = len(close)
    last, first = float(close.iloc[-1]), float(close.iloc[0])
    daily = close.pct_change().dropna()

    metrics = {
        "rows": n,
        "start_date": str(df.index[0])[:10],
        "end_date": str(df.index[-1])[:10],
        "last_close": _round(last),
        "period_return_pct": _round((last / first - 1) * 100 if first else None, 2),
        "return_1m_pct": _round((last / float(close.iloc[-22]) - 1) * 100, 2) if n > 22 else None,
        "period_high": _round(_num(df["High"].max())),
        "period_low": _round(_num(df["Low"].min())),
        "avg_volume": _round(_num(df["Volume"].mean()), 0),
        "sma_20": _round(float(close.tail(20).mean())) if n >= 20 else None,
        "sma_50": _round(float(close.tail(50).mean())) if n >= 50 else None,
        "annualized_vol_pct": _round(float(daily.std() * math.sqrt(252) * 100), 2) if len(daily) >= 2 else None,
    }
    history = [
        {"date": str(idx)[:10], **{c.lower(): _round(_num(row[c])) for c in PRICE_COLS}}
        for idx, row in df.iterrows()
    ]
    return ToolResult.success(tool, {"ticker": sym, "metrics": metrics, "history": history}, SOURCE, warnings)


def get_fundamentals(ticker: str, ticker_factory: Callable | None = None) -> ToolResult:
    tool, sym = "get_fundamentals", ticker.upper().strip()
    try:
        t = (ticker_factory or _default_factory)(sym)
        info = t.info or {}
    except Exception as exc:
        return ToolResult.failure(tool, f"{type(exc).__name__}: {exc}", SOURCE)

    fields = {k: _num(info.get(src)) for k, src in NUMERIC_FIELDS.items()}
    fields.update({k: (info.get(src) or None) for k, src in TEXT_FIELDS.items()})
    missing = [k for k, v in fields.items() if v is None]

    # Earnings date is optional: only report it if the provider has it.
    try:
        cal = getattr(t, "calendar", None)
        dates = cal.get("Earnings Date") if isinstance(cal, dict) else None
        fields["next_earnings_date"] = str(dates[0]) if dates else None
    except Exception:
        fields["next_earnings_date"] = None
    if fields["next_earnings_date"] is None:
        missing.append("next_earnings_date")

    if all(fields[k] is None for k in NUMERIC_FIELDS):
        return ToolResult.failure(tool, f"No fundamentals returned for '{sym}'", SOURCE,
                                  [f"Missing: {', '.join(missing)}"])
    warnings = [f"Missing fields: {', '.join(missing)}"] if missing else []
    return ToolResult.success(tool, {"ticker": sym, "fields": fields, "missing": missing}, SOURCE, warnings)


def to_memory_metrics(price: ToolResult | None, fundamentals: ToolResult | None) -> dict[str, Any]:
    """Flatten the numbers worth remembering between runs."""
    out: dict[str, Any] = {}
    if price and price.ok:
        m = price.data["metrics"]
        out.update({k: m.get(k) for k in ("last_close", "period_return_pct", "return_1m_pct",
                                          "sma_20", "sma_50", "annualized_vol_pct", "end_date")})
    if fundamentals and fundamentals.ok:
        f = fundamentals.data["fields"]
        out.update({k: f.get(k) for k in ("market_cap", "trailing_pe", "forward_pe", "trailing_eps")})
    return out


Writing market_tools.py


## Step 5 - News tools (NewsAPI / headline dump): This section of our agent project, its purpose is similar to market_tools.py, but instead of stock-market data, it retrieves recent news/headlines.

In [ ]:
%%writefile news_tools.py
"""News tools: NewsAPI (live) or a documented headline dump (JSON/CSV)."""
from __future__ import annotations

import csv
import json
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone
from typing import Callable

from schema import ToolResult

MAX_BODY = 500


def _default_get(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=15) as r:
        return json.loads(r.read().decode())


def _clean(a: dict) -> dict:
    src = a.get("source")
    return {
        "title": (a.get("title") or "").strip(),
        "body": (a.get("description") or a.get("body") or a.get("content") or "")[:MAX_BODY],
        "source": (src.get("name") if isinstance(src, dict) else src) or "unknown",
        "published_at": a.get("publishedAt") or a.get("published_at") or a.get("timestamp") or "",
        "url": a.get("url") or "",
    }


def _dedupe(items: list[dict]) -> list[dict]:
    seen, out = set(), []
    for it in items:
        key = it["title"].lower()
        if it["title"] and key not in seen:
            seen.add(key)
            out.append(it)
    return sorted(out, key=lambda x: x["published_at"], reverse=True)


def fetch_news_newsapi(query: str, api_key: str | None, days: int = 14, limit: int = 30,
                       http_get: Callable[[str], dict] | None = None) -> ToolResult:
    tool, src = "fetch_news", "NewsAPI"
    if not api_key and http_get is None:
        return ToolResult.failure(tool, "NEWSAPI_KEY not set", src)
    since = (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%d")
    qs = urllib.parse.urlencode({"q": query, "from": since, "language": "en", "sortBy": "publishedAt",
                                 "pageSize": limit, "apiKey": api_key or ""})
    try:
        payload = (http_get or _default_get)(f"https://newsapi.org/v2/everything?{qs}")
    except Exception as exc:
        return ToolResult.failure(tool, f"{type(exc).__name__}: {exc}", src)
    if payload.get("status") not in (None, "ok"):
        return ToolResult.failure(tool, payload.get("message", "NewsAPI error"), src)
    items = _dedupe([_clean(a) for a in payload.get("articles", [])])
    warnings = [] if items else ["No articles returned"]
    return ToolResult.success(tool, {"query": query, "count": len(items), "articles": items}, src, warnings)


def load_headline_dump(path: str) -> ToolResult:
    tool, src = "fetch_news", f"headline dump: {path}"
    try:
        if path.lower().endswith(".csv"):
            with open(path, newline="", encoding="utf-8") as f:
                raw = list(csv.DictReader(f))
        else:
            with open(path, encoding="utf-8") as f:
                raw = json.load(f)
            raw = raw.get("articles", raw) if isinstance(raw, dict) else raw
    except Exception as exc:
        return ToolResult.failure(tool, f"{type(exc).__name__}: {exc}", src)
    items = _dedupe([_clean(a) for a in raw])
    return ToolResult.success(tool, {"query": "dump", "count": len(items), "articles": items}, src,
                              [] if items else ["Dump contained no articles"])


Writing news_tools.py


Step 6 - Optional tools: FRED + SEC EDGAR: The agent that decides which tool to call and combines their results. These optional tools make it much more powerful because they provide official economic and company filing data.

All four tools can use the same ToolResult schema we created earlier. The schema.py is becomes the common interface that lets the agent work with completely different sources in a consistent way.

In [ ]:
%%writefile extra_tools.py
"""Optional data tools: FRED (macro) and SEC EDGAR (filings). Both accept an injectable http_get for offline tests."""
import json
import os
import urllib.request

from schema import ToolResult


def _get(url, headers=None):
    req = urllib.request.Request(url, headers=headers or {})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())


def _sec_get(url):
    ua = os.getenv("SEC_USER_AGENT", "InvestmentResearchAgent student-project contact@example.com")
    return _get(url, {"User-Agent": ua})


def get_macro(series=("FEDFUNDS", "DGS10", "CPIAUCSL"), api_key=None, http_get=None):
    tool, src = "get_macro", "FRED"
    key = api_key or os.getenv("FRED_API_KEY")
    if not key and http_get is None:
        return ToolResult.failure(tool, "FRED_API_KEY not set", src)
    out, warns = [], []
    for sid in series:
        url = (f"https://api.stlouisfed.org/fred/series/observations?series_id={sid}"
               f"&api_key={key or ''}&file_type=json&sort_order=desc&limit=3")
        try:
            obs = (http_get or _get)(url).get("observations", [])
        except Exception as exc:
            warns.append(f"{sid}: {type(exc).__name__}: {exc}")
            continue
        for o in obs:
            try:
                out.append({"id": sid, "date": o["date"], "value": float(o["value"])})
                break
            except (KeyError, ValueError, TypeError):
                continue  # FRED uses "." for missing values
    if not out:
        return ToolResult.failure(tool, "No FRED observations returned", src, warns)
    return ToolResult.success(tool, {"series": out}, src, warns)


def get_edgar_filings(ticker, forms=("10-K", "10-Q", "8-K"), limit=5, http_get=None):
    tool, src = "get_edgar_filings", "SEC EDGAR"
    get = http_get or _sec_get
    try:
        table = get("https://www.sec.gov/files/company_tickers.json")
        row = next((v for v in table.values() if v.get("ticker", "").upper() == ticker.upper()), None)
        if not row:
            return ToolResult.failure(tool, f"Ticker {ticker} not found in SEC ticker table", src)
        cik = str(row["cik_str"]).zfill(10)
        recent = get(f"https://data.sec.gov/submissions/CIK{cik}.json")["filings"]["recent"]
    except Exception as exc:
        return ToolResult.failure(tool, f"{type(exc).__name__}: {exc}", src)
    filings = []
    for form, d, acc, doc in zip(recent["form"], recent["filingDate"], recent["accessionNumber"],
                                 recent["primaryDocument"]):
        if form in forms:
            filings.append({"form": form, "date": d,
                            "url": f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-', '')}/{doc}"})
        if len(filings) >= limit:
            break
    warns = [] if filings else ["No matching recent filings"]
    return ToolResult.success(tool, {"company": row.get("title"), "cik": cik, "filings": filings}, src, warns)


Writing extra_tools.py


## Step 7 - LLM wrapper: An LLM wrapper is a small piece of code that provides a standard interface for talking to an LLM. In our agent project, it act as the bridge between your Python agent and the language model.

In [ ]:
%%writefile llm.py
"""LLM wrappers + tolerant JSON parsing.
Every LLM call in the project goes through a callable: (system, user) -> str.
Wrappers never raise: on API errors they return "" and the agent falls back to its rule-based logic."""
import json
import os
import re
import time


class ClaudeLLM:
    def __init__(self, model="claude-sonnet-5-5", max_tokens=1800):
        import anthropic

        self.client = anthropic.Anthropic()
        self.model, self.max_tokens, self.calls, self.last_error = model, max_tokens, 0, None

    def __call__(self, system: str, user: str) -> str:
        self.calls += 1
        try:
            r = self.client.messages.create(model=self.model, max_tokens=self.max_tokens, system=system,
                                            messages=[{"role": "user", "content": user}])
            return "".join(b.text for b in r.content if b.type == "text")
        except Exception as exc:
            self.last_error = f"{type(exc).__name__}: {exc}"
            return ""


class GeminiLLM:
    """Google Gemini via the `google-genai` SDK (pip install google-genai)."""

    def __init__(self, model="gemini-2.5-flash", max_tokens=4096, api_key=None, retries=3):
        from google import genai
        from google.genai import types

        key = api_key or os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
        if not key:
            raise ValueError("Set GEMINI_API_KEY (or GOOGLE_API_KEY)")
        self.client, self._types = genai.Client(api_key=key), types
        self.model, self.max_tokens, self.retries = model, max_tokens, retries
        self.calls, self.last_error = 0, None

    def __call__(self, system: str, user: str) -> str:
        self.calls += 1
        cfg = self._types.GenerateContentConfig(system_instruction=system, max_output_tokens=self.max_tokens)
        for attempt in range(1, self.retries + 1):
            try:
                r = self.client.models.generate_content(model=self.model, contents=user, config=cfg)
                return r.text or ""  # None when the reply is blocked/empty
            except Exception as exc:
                self.last_error = f"{type(exc).__name__}: {exc}"
                transient = any(x in self.last_error for x in ("429", "503", "UNAVAILABLE", "RESOURCE_EXHAUSTED"))
                if transient and attempt < self.retries:
                    time.sleep(5 * attempt)  # free tier rate limits: back off and retry
                    continue
                return ""
        return ""


def make_llm(provider="auto"):
    """provider: 'gemini' | 'claude' | 'auto' (Gemini first if its key is set). Returns None if no key."""
    if provider in ("gemini", "auto") and (os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")):
        return GeminiLLM()
    if provider in ("claude", "auto") and os.getenv("ANTHROPIC_API_KEY"):
        return ClaudeLLM()
    return None


def parse_json(text):
    """Extract the first JSON object/array from an LLM reply (handles ``` fences). None on failure."""
    if not text:
        return None
    text = re.sub(r"```(?:json)?", "", text)
    starts = [(text.find(o), c) for o, c in (("{", "}"), ("[", "]")) if text.find(o) != -1]
    for i, closer in sorted(starts):
        j = text.rfind(closer)
        if j > i:
            try:
                return json.loads(text[i:j + 1])
            except json.JSONDecodeError:
                pass
    return None


Writing llm.py


## Step 8 - Memory with lessons (learning across runs): This step adds memory to our agent, so the agent can retain useful information from one run and use it in a later run.The agent doesn't just remember data; it remembers lessons about what worked, what didn't, and what it should do differently next time for better decision making.

In [ ]:
%%writefile memory.py
"""Enduring memory: past runs + LESSONS the agent learned. Stored as JSON keyed by ticker."""
import json
import os
from datetime import datetime, timezone


def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


class MemoryStore:
    def __init__(self, path="memory/store.json"):
        self.path = path
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    def _load(self):
        try:
            with open(self.path, encoding="utf-8") as f:
                return json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            return {}

    def _save(self, db):
        tmp = self.path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(db, f, indent=2)
        os.replace(tmp, self.path)

    def _entry(self, db, ticker):
        e = db.setdefault(ticker.upper(), {})
        e.setdefault("runs", [])
        e.setdefault("lessons", [])
        return e

    # ---- runs ----
    def last_run(self, ticker):
        runs = self._load().get(ticker.upper(), {}).get("runs", [])
        return runs[-1] if runs else None

    def save_run(self, ticker, run_id, metrics, headlines, critique, brief, scores=None):
        db = self._load()
        e = self._entry(db, ticker)
        e["runs"].append({"run_id": run_id, "saved_at": _now(), "metrics": metrics,
                          "headlines": headlines[:15], "critique": critique, "scores": scores or {},
                          "brief": brief})
        e["runs"] = e["runs"][-20:]
        self._save(db)

    @staticmethod
    def diff(prev, current):
        if not prev:
            return []
        out = []
        for k, new in current.items():
            old = prev.get("metrics", {}).get(k)
            if isinstance(new, (int, float)) and isinstance(old, (int, float)) and old != new:
                pct = f" ({(new / old - 1) * 100:+.1f}%)" if old else ""
                out.append(f"{k}: {old} -> {new}{pct}")
        return out

    # ---- lessons (learning across runs) ----
    def get_lessons(self, ticker):
        return self._load().get(ticker.upper(), {}).get("lessons", [])

    def update_lessons(self, ticker, rule_lessons, llm_lessons):
        """Rule-based lessons are REPLACED each run (they describe the latest run);
        LLM lessons accumulate (max 5 newest)."""
        db = self._load()
        e = self._entry(db, ticker)
        kept_llm = [l for l in e["lessons"] if l.get("source") == "llm"]
        new_llm = [{"tag": "llm", "text": t, "source": "llm", "updated": _now()} for t in llm_lessons]
        seen, merged = set(), []
        for l in new_llm + kept_llm:
            if l["text"] not in seen:
                seen.add(l["text"])
                merged.append(l)
        rules = [{**l, "source": "rules", "updated": _now()} for l in rule_lessons]
        e["lessons"] = rules + merged[:5]
        self._save(db)
        return e["lessons"]


Writing memory.py


### Step 9 - PATTERN 1: Prompt chaining (Ingest -> Preprocess -> Classify -> Extract -> Summarize). This step will break a complex task into smaller sequential prompts, with each step producing information that the next step uses. Prompt chaining is especially useful when:the task has multiple logical stages, provides more predictable outputs, need structured information from unstructured text, evaluate each step separately and identify where an error occurred.

In [ ]:
%%writefile chain.py
"""WORKFLOW PATTERN 1 - PROMPT CHAINING
Ingest -> Preprocess -> Classify -> Extract -> Summarize
Each stage consumes the previous stage's output. A programmatic GATE validates every stage; if LLM output
fails the gate, that item falls back to deterministic rules (so the chain never breaks)."""
import html
import json
import re
from collections import Counter

from llm import parse_json

CATEGORIES = ["earnings", "product", "legal_regulatory", "macro", "analyst_rating", "m_and_a", "other"]
SENTIMENTS = ["positive", "neutral", "negative"]
KEYWORDS = {
    "earnings": ["earnings", "eps", "revenue", "guidance", "quarter", "quarterly", "profit", "results"],
    "analyst_rating": ["upgrade", "downgrade", "price target", "analyst", "rating", "outperform", "underperform"],
    "legal_regulatory": ["lawsuit", "sues", "antitrust", "regulator", "regulatory", "fine", "sec", "doj", "ftc", "probe", "court"],
    "m_and_a": ["acquire", "acquisition", "merger", "buyout", "stake", "takeover"],
    "macro": ["fed", "rate cut", "rate hike", "inflation", "tariff", "tariffs", "gdp", "recession", "cpi"],
    "product": ["launch", "unveil", "release", "announces", "new product", "feature", "rollout"],
}
POS = ["beat", "surge", "record", "growth", "upgrade", "gain", "rally", "strong", "raises", "soar", "profit", "outperform"]
NEG = ["miss", "fall", "drop", "lawsuit", "downgrade", "cut", "weak", "plunge", "probe", "fine", "decline",
       "slump", "concern", "risk", "underperform", "ban"]

CLASSIFY_SYSTEM = (
    "You classify financial news. For each article return the category "
    f"(one of {CATEGORIES}) and the sentiment for the stock (one of {SENTIMENTS}). "
    'Return JSON only: [{"id":"N1","category":"...","sentiment":"..."}]')
EXTRACT_SYSTEM = (
    "You extract facts from financial news. For each article give up to 3 concrete facts (events, figures, "
    'dates) copied from the text. Return JSON only: [{"id":"N1","facts":["..."],"numbers":["..."]}]')
SUMMARY_SYSTEM = (
    "You write news summaries for an equity analyst. Using ONLY the provided classified facts, write 3-4 "
    "sentences. Put the article id in brackets, like [N3], after every claim. Return plain text.")


def _has(text, kw):
    pat = rf"\b{re.escape(kw)}\w*" if len(kw) >= 5 else rf"\b{re.escape(kw)}\b"
    return re.search(pat, text) is not None


def _clean(s):
    s = html.unescape(re.sub(r"<[^>]+>", " ", s or ""))
    return re.sub(r"\s+", " ", s).strip()


# ---- Stage 1: INGEST ----------------------------------------------------
def ingest(news_results):
    arts, errors = [], []
    for r in news_results:
        if r.ok:
            arts += r.data.get("articles", [])
        else:
            errors.append(f"{r.source or r.tool}: {r.error}")
    return arts, errors


# ---- Stage 2: PREPROCESS --------------------------------------------------
def preprocess(articles, ticker, company=None, max_articles=30):
    names = [ticker.lower()] + ([company.lower().split()[0]] if company else [])
    seen, out = set(), []
    for a in articles:
        title, body = _clean(a.get("title")), _clean(a.get("body"))
        key = re.sub(r"\W+", "", title.lower())
        if not title or key in seen:
            continue
        seen.add(key)
        text = f"{title} {body}".lower()
        out.append({"title": title, "body": body[:400], "source": a.get("source") or "unknown",
                    "published_at": a.get("published_at", ""), "url": a.get("url", ""),
                    "relevant": any(_has(text, n) for n in names)})
    relevant = [a for a in out if a["relevant"]]
    kept = (relevant if len(relevant) >= 3 else out)[:max_articles]  # gate: never filter down to nothing
    for i, a in enumerate(kept, 1):
        a["id"] = f"N{i}"
    return kept, {"in": len(articles), "after_dedupe": len(out), "kept": len(kept)}


# ---- Stage 3: CLASSIFY ----------------------------------------------------
def _rule_classify(a):
    text = f"{a['title']} {a['body']}".lower()
    hits = {c: sum(_has(text, k) for k in kws) for c, kws in KEYWORDS.items()}
    cat = max(hits, key=hits.get) if any(hits.values()) else "other"
    score = sum(_has(text, w) for w in POS) - sum(_has(text, w) for w in NEG)
    return cat, ("positive" if score > 0 else "negative" if score < 0 else "neutral")


def classify(articles, llm=None):
    labels = {}
    if llm:
        payload = [{"id": a["id"], "title": a["title"], "body": a["body"][:200]} for a in articles]
        data = parse_json(llm(CLASSIFY_SYSTEM, json.dumps(payload)))
        if isinstance(data, list):
            labels = {d.get("id"): d for d in data if isinstance(d, dict)}
    for a in articles:
        p = labels.get(a["id"], {})
        if p.get("category") in CATEGORIES and p.get("sentiment") in SENTIMENTS:  # GATE
            a["category"], a["sentiment"], a["label_source"] = p["category"], p["sentiment"], "llm"
        else:
            a["category"], a["sentiment"] = _rule_classify(a)
            a["label_source"] = "rules"
    return articles


# ---- Stage 4: EXTRACT -----------------------------------------------------
NUM_RE = re.compile(r"\$?\d[\d,]*(?:\.\d+)?\s?(?:%|percent|billion|million|trillion|bn)?", re.I)


def extract(articles, llm=None):
    parsed = {}
    if llm:
        payload = [{"id": a["id"], "title": a["title"], "body": a["body"]} for a in articles]
        data = parse_json(llm(EXTRACT_SYSTEM, json.dumps(payload)))
        if isinstance(data, list):
            parsed = {d.get("id"): d for d in data if isinstance(d, dict)}
    for a in articles:
        d = parsed.get(a["id"], {})
        facts = [str(x)[:200] for x in d["facts"] if isinstance(x, str)][:3] if isinstance(d.get("facts"), list) else []
        nums = [str(x)[:40] for x in d["numbers"] if isinstance(x, (str, int, float))][:5] if isinstance(d.get("numbers"), list) else []
        if facts:  # GATE: need at least one fact
            a["extract_source"] = "llm"
        else:
            facts = [a["title"]]
            nums = [n.strip() for n in NUM_RE.findall(f"{a['title']} {a['body']}") if len(n.strip()) >= 2][:5]
            a["extract_source"] = "rules"
        a["facts"], a["numbers"] = facts, nums
    return articles


# ---- Stage 5: SUMMARIZE ---------------------------------------------------
def summarize(articles, llm=None):
    sc = Counter(a["sentiment"] for a in articles)
    cc = Counter(a["category"] for a in articles)
    top = sorted(articles, key=lambda a: a["sentiment"] == "neutral")[:3]
    themes = ", ".join(f"{k} ({v})" for k, v in cc.most_common(3))
    notable = "; ".join(f"{a['title']} [{a['id']}]" for a in top)
    text = (f"{len(articles)} relevant articles: {sc['positive']} positive, {sc['neutral']} neutral, "
            f"{sc['negative']} negative. Main themes: {themes}. Notable: {notable}")
    src = "rules"
    if llm:
        payload = [{"id": a["id"], "category": a["category"], "sentiment": a["sentiment"], "facts": a["facts"]}
                   for a in articles[:15]]
        out = (llm(SUMMARY_SYSTEM, json.dumps(payload)) or "").strip()
        cited = set(re.findall(r"\[(N\d+)\]", out))
        if out and cited and cited <= {a["id"] for a in articles}:  # GATE: must cite real articles
            text, src = out, "llm"
    return {"summary": text, "summary_source": src, "sentiment_counts": dict(sc), "category_counts": dict(cc)}


# ---- Orchestration --------------------------------------------------------
def run_news_chain(news_results, ticker, company=None, llm=None):
    log = []
    arts, errors = ingest(news_results)
    log.append({"stage": "ingest", "articles": len(arts), "errors": errors})
    arts, pstats = preprocess(arts, ticker, company)
    log.append({"stage": "preprocess", **pstats})
    if not arts:
        return {"articles": [], "summary": "No usable news articles were retrieved.", "sentiment_counts": {},
                "category_counts": {}, "stage_log": log, "errors": errors}
    arts = classify(arts, llm)
    log.append({"stage": "classify", "by_llm": sum(a["label_source"] == "llm" for a in arts),
                "by_rules": sum(a["label_source"] == "rules" for a in arts)})
    arts = extract(arts, llm)
    log.append({"stage": "extract", "by_llm": sum(a["extract_source"] == "llm" for a in arts),
                "by_rules": sum(a["extract_source"] == "rules" for a in arts)})
    digest = summarize(arts, llm)
    log.append({"stage": "summarize", "source": digest["summary_source"]})
    return {"articles": arts, **digest, "stage_log": log, "errors": errors}


Writing chain.py


## Step 10 - PATTERN 2: Routing to specialists: Routing to specialists means having one main/orchestrator agent chooses the path and decide which specialized agent or tool should handle a particular request. The main agent acts like a coordinator: it analyzes the user's request and sends it to the specialist, tool, or workflow best suited to handle that type of task.

In [ ]:
%%writefile router.py
"""WORKFLOW PATTERN 2 - ROUTING
Each piece of content (tool result or classified news article) is directed to the right specialist:
  market   -> price action, technicals, macro backdrop
  earnings -> fundamentals, valuation, filings, earnings/analyst news
  news     -> all other news (product, legal, macro, M&A, other)
The classifier output from the prompt chain (article category) drives news routing."""
from datetime import date

TOOL_ROUTES = {"get_price_history": "market", "get_macro": "market",
               "get_fundamentals": "earnings", "get_edgar_filings": "earnings"}
ARTICLE_ROUTES = {"earnings": "earnings", "analyst_rating": "earnings"}  # everything else -> news


def _cite(*ids):
    return " " + "".join(f"[{i}]" for i in ids)


def _fmt(v, nd=2, suf=""):
    return "n/a" if v is None else f"{v:.{nd}f}{suf}"


def _fmt_big(v):
    if v is None:
        return "n/a"
    for div, suf in ((1e12, "T"), (1e9, "B"), (1e6, "M")):
        if abs(v) >= div:
            return f"${v / div:.2f}{suf}"
    return f"${v:,.0f}"


def _find(items, tool):
    return next((i for i in items if i["kind"] == "tool" and i["tool"] == tool and i["result"].ok), None)


def _days_until(datestr, today):
    try:
        return (date.fromisoformat(str(datestr)[:10]) - date.fromisoformat(today)).days
    except Exception:
        return None


class Router:
    def route(self, items):
        buckets, log = {"market": [], "earnings": [], "news": []}, []
        for it in items:
            if it["kind"] == "tool":
                dest, why = TOOL_ROUTES.get(it["tool"]), f"tool '{it['tool']}'"
            else:
                cat = it["article"]["category"]
                dest, why = ARTICLE_ROUTES.get(cat, "news"), f"article category '{cat}'"
            if dest is None:
                log.append({"item": it["sid"], "to": None, "why": why + " has no specialist"})
                continue
            buckets[dest].append(it)
            log.append({"item": it["sid"], "to": dest, "why": why})
        return buckets, log


class MarketSpecialist:
    def analyze(self, items, ctx):
        out = {"headline": "Market data unavailable.", "findings": [], "risks": []}
        price, macro = _find(items, "get_price_history"), _find(items, "get_macro")
        if price:
            m, c = price["result"].data["metrics"], _cite(price["sid"])
            last, s20, s50 = m["last_close"], m.get("sma_20"), m.get("sma_50")
            pos = None
            if s20 and s50:
                pos = "above both" if last > s20 and last > s50 else "below both" if last < s20 and last < s50 else "between"
            hi, lo = m.get("period_high"), m.get("period_low")
            rng = (last - lo) / (hi - lo) * 100 if hi and lo and hi > lo else None
            out["findings"] = [
                f"Last close {last} on {m['end_date']}{c}",
                f"Return {_fmt(m['period_return_pct'], 1, '%')} over {m['rows']} sessions; last month {_fmt(m.get('return_1m_pct'), 1, '%')}{c}",
                f"20-day average {_fmt(s20)} and 50-day average {_fmt(s50)}; price is {pos or 'n/a'} of them{c}",
                f"Sits at {_fmt(rng, 0, '%')} of its period range ({lo} to {hi}){c}",
                f"Annualized volatility {_fmt(m.get('annualized_vol_pct'), 1, '%')}{c}",
            ]
            out["headline"] = (f"{ctx['ticker']} closed at {last}, {_fmt(m['period_return_pct'], 1, '%')} over "
                               f"{m['rows']} sessions, trading {pos or 'n/a'} of its 20/50-day averages{c}")
            vol = m.get("annualized_vol_pct")
            if vol and vol > 35:
                out["risks"].append(f"Elevated volatility ({vol:.1f}% annualized){c}")
            if pos == "below both":
                out["risks"].append(f"Price is below its 20- and 50-day averages (weak trend){c}")
            if rng is not None and rng > 90:
                out["risks"].append(f"Trading near the top of its period range ({rng:.0f}%), limited cushion{c}")
        if macro:
            c = _cite(macro["sid"])
            for s in macro["result"].data["series"]:
                out["findings"].append(f"Macro: {s['id']} = {s['value']} ({s['date']}){c}")
        return out


class EarningsSpecialist:
    def analyze(self, items, ctx):
        out = {"headline": "Fundamentals unavailable.", "findings": [], "risks": []}
        fund, edgar = _find(items, "get_fundamentals"), _find(items, "get_edgar_filings")
        if fund:
            f, c = fund["result"].data["fields"], _cite(fund["sid"])
            tpe, fpe = f.get("trailing_pe"), f.get("forward_pe")
            out["findings"] += [
                f"{f.get('name') or ctx['ticker']} ({f.get('sector') or 'sector n/a'}), market cap {_fmt_big(f.get('market_cap'))}{c}",
                f"Trailing P/E {_fmt(tpe, 1)}, forward P/E {_fmt(fpe, 1)}, trailing EPS {_fmt(f.get('trailing_eps'))}{c}"]
            if tpe and fpe:
                out["findings"].append(("Forward P/E below trailing implies expected earnings growth" if fpe < tpe
                                        else "Forward P/E above trailing implies expected earnings decline") + c)
            days = _days_until(f.get("next_earnings_date"), ctx["run_date"])
            if f.get("next_earnings_date"):
                out["findings"].append(f"Next earnings {str(f['next_earnings_date'])[:10]}"
                                       + (f" (in {days} days)" if days is not None else "") + c)
            if days is not None and 0 <= days <= 14:
                out["risks"].append(f"Earnings in {days} days: event risk{c}")
            if tpe and tpe > 30:
                out["risks"].append(f"Premium valuation (trailing P/E {tpe:.1f}) leaves less room for disappointment{c}")
            out["headline"] = (f"Valued at {_fmt(tpe, 1)}x trailing earnings ({_fmt(fpe, 1)}x forward); "
                               f"next earnings {str(f.get('next_earnings_date') or 'n/a')[:10]}{c}")
        if edgar:
            fl = edgar["result"].data["filings"]
            out["findings"].append("Recent filings: " + "; ".join(f"{x['form']} {x['date']}" for x in fl[:4])
                                   + _cite(edgar["sid"]))
        for it in items:
            if it["kind"] == "article":
                a, c = it["article"], _cite(it["sid"])
                out["findings"].append(f"{a['category'].replace('_', ' ').title()} ({a['sentiment']}): {a['title']}{c}")
                if a["sentiment"] == "negative":
                    out["risks"].append(f"Negative earnings/analyst news: {a['title']}{c}")
        return out


class NewsSpecialist:
    def analyze(self, items, ctx):
        arts = [i for i in items if i["kind"] == "article"]
        out = {"headline": "No news retrieved.", "findings": [], "risks": []}
        if not arts:
            return out
        pos = sum(i["article"]["sentiment"] == "positive" for i in arts)
        neg = sum(i["article"]["sentiment"] == "negative" for i in arts)
        tone = "positive" if pos > neg else "negative" if neg > pos else "mixed"
        ids = [i["sid"] for i in arts[:3]]
        out["headline"] = f"Non-earnings news tone is {tone} ({pos} positive vs {neg} negative of {len(arts)})" + _cite(*ids)
        for it in arts[:5]:
            a, c = it["article"], _cite(it["sid"])
            out["findings"].append(f"{a['title']} ({a['source']}, {a['sentiment']}){c}")
            if a["sentiment"] == "negative":
                out["risks"].append(f"Negative headline: {a['title']}{c}")
        return out


SPECIALISTS = {"market": MarketSpecialist(), "earnings": EarningsSpecialist(), "news": NewsSpecialist()}


def run_specialists(buckets, ctx):
    return {name: SPECIALISTS[name].analyze(items, ctx) for name, items in buckets.items()}


Writing router.py


## Step 11 - PATTERN 3: Evaluator-Optimizer: When one agent does the work; another checks the work and provides feedback so the work can be improved.

In [ ]:
%%writefile evaluator.py
"""WORKFLOW PATTERN 3 - EVALUATOR-OPTIMIZER
generate draft -> evaluate against a rubric -> refine using the feedback -> re-evaluate (loop until pass or max rounds).
Evaluator = deterministic rubric checks (+ optional LLM judge). Optimizer = LLM rewrite (or rule-based fixer offline)."""
import json
import re

from llm import parse_json

CRITERIA = ["grounding", "completeness", "balance", "transparency", "memory_use"]
NAME_MAP = {  # heading prefix (lowercase) -> canonical heading
    "summary": "Summary", "market": "Market", "earnings": "Earnings & Fundamentals", "news": "News",
    "changes since last run": "Changes since last run", "risks": "Risks & Open Questions",
    "data gaps": "Data gaps", "caveats": "Caveats"}
ORDER = list(NAME_MAP.values())
LEAN_SKIP = ("Risks & Open Questions", "Data gaps")  # first draft only writes sections with computed findings
CITE_RE = re.compile(r"\[((?:S|N)\d+)\]")

WRITER_SYSTEM = (
    "You are an equity research analyst. Write a SHORT analyst note in Markdown using ONLY the evidence JSON. "
    "Sections in this order: Summary, Market, Earnings & Fundamentals, News, Changes since last run, "
    "Risks & Open Questions, Data gaps, Caveats. Cite every figure with ids from valid_citation_ids, like [S2] or "
    "[N3]. Never invent ids. State missing data explicitly. No bare BUY/SELL rating. Apply lessons_from_past_runs.")
JUDGE_SYSTEM = (
    "You are a strict reviewer of analyst notes. Score 1-5 on: grounding (every number cited to real evidence), "
    "completeness, balance (risks stated), transparency (data gaps disclosed), memory_use (compares with prior run). "
    'Return JSON only: {"scores":{"grounding":5,...},"feedback":["specific fix", ...]}')
REFINE_SYSTEM = (
    "You revise analyst notes. Rewrite the note so that EVERY feedback item is fixed. Keep the same Markdown "
    "sections and [S#]/[N#] citations; use only the evidence provided; do not invent facts. Return the full note.")


def compact(ctx):
    return {"ticker": ctx["ticker"], "date": ctx["run_date"], "specialists": ctx["specialists"],
            "news_summary": ctx["digest"]["summary"], "changes": ctx["changes"], "gaps": ctx["gaps"],
            "lessons_from_past_runs": ctx["lessons"], "valid_citation_ids": ctx["valid_ids"],
            "sources": {k: {"tool": v["tool"], "source": v["source"], "fetched_at": v["fetched_at"]}
                        for k, v in ctx["sources"].items()}}


# ---------- template writer ----------
def _bullets(xs):
    return [f"- {x}" for x in xs]


def build_sections(ctx):
    sp = ctx["specialists"]
    cite = f" [{ctx['memory_sid']}]" if ctx.get("memory_sid") else ""
    if ctx["changes"]:
        changes = _bullets([c + cite for c in ctx["changes"]])
    elif ctx["prior"]:
        changes = [f"- No change in tracked metrics since run {ctx['prior']['run_id']}{cite}"]
    else:
        changes = ["- No prior run to compare."]
    risks = [r for k in ("market", "earnings", "news") for r in sp[k]["risks"]]
    return {
        "Summary": _bullets([sp[k]["headline"] for k in ("market", "earnings", "news")]),
        "Market": _bullets(sp["market"]["findings"]) or ["- Market data unavailable."],
        "Earnings & Fundamentals": _bullets(sp["earnings"]["findings"]) or ["- Fundamentals unavailable."],
        "News": [ctx["digest"]["summary"], *_bullets(sp["news"]["findings"])],
        "Changes since last run": changes,
        "Risks & Open Questions": _bullets(risks) or ["- No material risks flagged by the specialists (absence of evidence is not proof of safety)."],
        "Data gaps": _bullets(ctx["gaps"]) or ["- None detected."],
        "Caveats": ["- Automated summary of public data; not investment advice. Figures are only as fresh as the fetch times in the audit log."],
    }


def compose(ctx, skip=()):
    secs = build_sections(ctx)
    lines = [f"# {ctx['ticker']} Analyst Note ({ctx['run_date']})", ""]
    for name in ORDER:
        if name not in skip:
            lines += [f"## {name}", *secs[name], ""]
    return "\n".join(lines).rstrip() + "\n"


def generate_draft(ctx, llm=None):
    if llm:
        out = (llm(WRITER_SYSTEM, json.dumps(compact(ctx), default=str)) or "").strip()
        if len(out) > 200:
            return out + "\n"
    learned = {"weak:balance", "weak:transparency", "weak:completeness"} & set(ctx["lesson_tags"])
    return compose(ctx, skip=() if learned else LEAN_SKIP)  # LEARNING: past lessons change the first draft


# ---------- evaluator ----------
def sections(brief):
    parts, cur = {}, None
    for line in brief.splitlines():
        m = re.match(r"##\s+(.+)", line)
        if m:
            cur = m.group(1).strip().lower()
            parts[cur] = []
        elif cur is not None:
            parts[cur].append(line)
    return parts


def _has(parts, prefix):
    return any(k.startswith(prefix) for k in parts)


def _get(parts, prefix):
    return next((v for k, v in parts.items() if k.startswith(prefix)), [])


def rule_evaluate(brief, ctx):
    parts, fb, sc = sections(brief), [], {}
    valid, bad, numeric, cited = set(ctx["valid_ids"]), set(), 0, 0
    for name, lines in parts.items():
        if name.startswith(("data gaps", "caveats")):
            continue
        for ln in lines:
            ids = CITE_RE.findall(ln)
            bad |= {i for i in ids if i not in valid}
            if re.search(r"\d", ln):
                numeric += 1
                cited += bool(ids)
    ratio = cited / numeric if numeric else 1.0
    sc["grounding"] = round(1 + 4 * ratio)
    if ratio < 1:
        fb.append(f"{numeric - cited} line(s) containing numbers have no [S#]/[N#] citation.")
    if bad:
        sc["grounding"] = min(sc["grounding"], 2)
        fb.append(f"Remove citations to unknown sources: {sorted(bad)}.")

    missing = [k for k in NAME_MAP if not _has(parts, k)]
    sc["completeness"] = round(1 + 4 * (len(NAME_MAP) - len(missing)) / len(NAME_MAP))
    if missing:
        fb.append("Missing sections: " + ", ".join(NAME_MAP[k] for k in missing) + ".")

    risk_bullets = [l for l in _get(parts, "risks") if l.strip().startswith("-")]
    sc["balance"] = 5 if risk_bullets else 2
    if not risk_bullets:
        fb.append("Add a 'Risks & Open Questions' section with at least one bullet.")

    gtext = " ".join(_get(parts, "data gaps")).lower()
    if ctx["gaps"]:
        undisclosed = [g for g in ctx["gaps"] if g.split(":")[0].lower() not in gtext]
        sc["transparency"] = 5 if _has(parts, "data gaps") and not undisclosed else 2
        if sc["transparency"] < 5:
            fb.append("Disclose every data gap: " + "; ".join(ctx["gaps"]))
    else:
        sc["transparency"] = 5 if _has(parts, "data gaps") else 4

    ctext = " ".join(_get(parts, "changes since last run")).lower()
    sc["memory_use"] = 2 if ctx["prior"] and ("no prior run" in ctext or not ctext.strip()) else 5
    if sc["memory_use"] < 5:
        fb.append("A prior run exists: report what changed since it (metrics from memory).")
    return {"scores": sc, "feedback": fb}


def evaluate(brief, ctx, llm=None, threshold=4.0):
    ev = rule_evaluate(brief, ctx)
    if llm:
        j = parse_json(llm(JUDGE_SYSTEM, json.dumps({"note": brief, "evidence": compact(ctx)}, default=str)))
        if isinstance(j, dict) and isinstance(j.get("scores"), dict):
            for k in CRITERIA:
                v = j["scores"].get(k)
                if isinstance(v, (int, float)) and 1 <= v <= 5:
                    ev["scores"][k] = min(ev["scores"][k], round(v))
            ev["feedback"] += [x for x in j.get("feedback", []) if isinstance(x, str)][:5]
    ev["overall"] = round(sum(ev["scores"].values()) / len(CRITERIA), 2)
    ev["passed"] = ev["overall"] >= threshold and min(ev["scores"].values()) >= 3
    return ev


# ---------- optimizer ----------
def _insert_missing(brief, ctx):
    parts, secs = sections(brief), build_sections(ctx)
    add = [t for k, t in NAME_MAP.items() if not _has(parts, k)]
    if not add:
        return brief
    block = "".join(f"## {t}\n" + "\n".join(secs[t]) + "\n\n" for t in add if t != "Caveats")
    out = brief.rstrip("\n") + "\n"
    if "## Caveats" in out:
        return out.replace("## Caveats", block + "## Caveats", 1)
    tail = "## Caveats\n" + "\n".join(secs["Caveats"]) + "\n" if "Caveats" in add else ""
    return out + "\n" + block + tail


def refine(brief, feedback, ctx, llm=None):
    if llm:
        out = (llm(REFINE_SYSTEM, json.dumps({"note": brief, "feedback": feedback, "evidence": compact(ctx)},
                                              default=str)) or "").strip()
        if len(out) > 200:
            return out + "\n"
    return _insert_missing(brief, ctx)  # offline fallback: synthesize the missing sections from specialist output


def optimize(ctx, llm=None, max_iter=3, threshold=4.0):
    brief, history = generate_draft(ctx, llm), []
    for i in range(max_iter + 1):
        ev = evaluate(brief, ctx, llm, threshold)
        history.append({"iteration": i, **ev, "chars": len(brief)})
        if ev["passed"] or i == max_iter:
            break
        new = refine(brief, ev["feedback"], ctx, llm)
        if new.strip() == brief.strip():
            break  # optimizer made no progress
        brief = new
    return {"brief": brief, "history": history, "passed": history[-1]["passed"]}


Writing evaluator.py


## Step 11b - The agent (plan -> act -> chain -> route -> optimize -> reflect):This step will help agent plan what to do, acts using tools, chains tasks together, routes work to specialists, optimizes the result through evaluation, and reflects on what it learned.

In [ ]:
%%writefile research_agent.py
"""Autonomous Investment Research Agent.
PLAN -> ACT (dynamic tools, retry, re-plan) -> CHAIN (news) -> ROUTE (specialists) -> EVALUATE/OPTIMIZE -> REFLECT & LEARN."""
import json
import os
import uuid
from datetime import date, datetime, timezone

import extra_tools as ex
import market_tools as mt
import news_tools as nt
from chain import run_news_chain
from evaluator import optimize
from llm import parse_json
from memory import MemoryStore
from router import TOOL_ROUTES, Router, run_specialists
from schema import ToolResult

MANDATORY = ["get_price_history", "get_fundamentals", "fetch_news"]
PLAN_SYSTEM = (
    "You are the planning module of an investment research agent. Choose which tools to call, in order, for the "
    "ticker. Use the lessons from past runs. Return JSON only: "
    '{"steps":[{"tool":"<name from catalog>","args":{},"why":"<short reason>"}]}')
REFLECT_SYSTEM = (
    "You are the agent reviewing your own run. Given the evaluation history and data gaps, return JSON only: "
    '{"assessment":"2 sentences on quality","lessons":["up to 3 short imperative lessons for the next run"]}')


def default_tools(news_file=None):
    def news(t, query=None):
        if news_file:
            return nt.load_headline_dump(news_file)
        return nt.fetch_news_newsapi(query or t, os.getenv("NEWSAPI_KEY"))

    return {
        "get_price_history": {"fn": lambda t, period="6mo": mt.get_price_history(t, period=period),
                              "desc": "Daily OHLCV + trend/volatility metrics. args: period (e.g. 3mo, 6mo, 1y)",
                              "args": ["period"]},
        "get_fundamentals": {"fn": lambda t: mt.get_fundamentals(t),
                             "desc": "Market cap, P/E, EPS, next earnings date", "args": []},
        "fetch_news": {"fn": news, "desc": "Recent news articles. args: query (defaults to ticker)", "args": ["query"]},
        "get_macro": {"fn": lambda t: ex.get_macro(), "desc": "FRED macro indicators (needs FRED_API_KEY)", "args": []},
        "get_edgar_filings": {"fn": lambda t: ex.get_edgar_filings(t),
                              "desc": "Recent SEC filings (10-K/10-Q/8-K)", "args": []},
    }


class ResearchAgent:
    def __init__(self, ticker, llm=None, memory=None, tools=None, news_file=None,
                 log_dir="runs", max_refinements=3, threshold=4.0):
        self.ticker, self.llm = ticker.upper().strip(), llm
        self.memory = memory or MemoryStore()
        self.news_file, self.log_dir = news_file, log_dir
        self.max_refinements, self.threshold = max_refinements, threshold
        self.tools = dict(tools or default_tools(news_file))
        self.tools["recall_memory"] = {"fn": self._recall, "args": [],
                                       "desc": "RETRIEVAL: last run's metrics and lessons learned from memory"}
        self.trace = []

    # ---------- tools ----------
    def _recall(self, ticker):
        prior, lessons = self.memory.last_run(ticker), self.memory.get_lessons(ticker)
        return ToolResult.success("recall_memory", {
            "prior_run_id": prior["run_id"] if prior else None,
            "metrics": prior["metrics"] if prior else {}, "lessons": [l["text"] for l in lessons]}, "local memory")

    def _call(self, step):
        name, spec = step["tool"], self.tools.get(step["tool"])
        if not spec:
            return ToolResult.failure(name, "unknown tool")
        args = {k: v for k, v in step.get("args", {}).items() if k in spec.get("args", [])}

        def once():
            try:
                return spec["fn"](self.ticker, **args)
            except Exception as exc:
                return ToolResult.failure(name, f"{type(exc).__name__}: {exc}")

        res, retried = once(), False
        if not res.ok and not any(x in (res.error or "") for x in ("not set", "401", "403", "unknown")):
            res, retried = once(), True  # one retry for transient errors
        self.trace.append({"phase": "tool", "tool": name, "args": args, "ok": res.ok, "error": res.error,
                           "retried": retried, "why": step.get("why", "")})
        return res

    # ---------- 1. PLAN ----------
    def plan(self):
        prior, lessons = self.memory.last_run(self.ticker), self.memory.get_lessons(self.ticker)
        steps, mode = None, "rules"
        if self.llm:
            payload = {"ticker": self.ticker, "tools": {k: v["desc"] for k, v in self.tools.items()},
                       "lessons": [l["text"] for l in lessons], "has_prior_run": bool(prior)}
            parsed = parse_json(self.llm(PLAN_SYSTEM, json.dumps(payload)))
            parsed = parsed.get("steps") if isinstance(parsed, dict) else parsed
            if isinstance(parsed, list):
                steps = [{"tool": s["tool"], "args": s["args"] if isinstance(s.get("args"), dict) else {},
                          "why": s.get("why", "")} for s in parsed if isinstance(s, dict) and s.get("tool") in self.tools]
                mode = "llm" if steps else "rules"
        if not steps:
            steps = self._rule_plan(prior, lessons)
        names = [s["tool"] for s in steps]
        for m in MANDATORY:  # safeguard: the note cannot be written without these
            if m not in names:
                steps.append({"tool": m, "args": {}, "why": "mandatory tool added by safeguard"})
        if "recall_memory" not in names:
            steps.insert(0, {"tool": "recall_memory", "args": {}, "why": "load prior metrics and lessons"})
        return steps, mode

    def _rule_plan(self, prior, lessons):
        tags = {l["tag"] for l in lessons}
        news_args = {}
        company = (prior or {}).get("metrics", {}).get("company")
        if "news_thin" in tags and company:  # LEARNED from last run
            news_args = {"query": f"{company} stock"}
        steps = [{"tool": "recall_memory", "args": {}, "why": "load prior metrics and lessons"},
                 {"tool": "get_price_history", "args": {}, "why": "price trend, volatility, moving averages"},
                 {"tool": "get_fundamentals", "args": {}, "why": "valuation and earnings date"},
                 {"tool": "fetch_news", "args": news_args, "why": "recent headlines" + (" (widened query, learned)" if news_args else "")}]
        if os.getenv("FRED_API_KEY") and "get_macro" in self.tools:
            steps.append({"tool": "get_macro", "args": {}, "why": "macro backdrop (FRED key present)"})
        if os.getenv("SEC_USER_AGENT") and "get_edgar_filings" in self.tools:
            steps.append({"tool": "get_edgar_filings", "args": {}, "why": "recent SEC filings"})
        return steps

    # ---------- 2. ACT (adaptive) ----------
    @staticmethod
    def _company(results, prior):
        for r in results:
            if r.tool == "get_fundamentals" and r.ok and r.data["fields"].get("name"):
                return r.data["fields"]["name"]
        return (prior or {}).get("metrics", {}).get("company")

    def _followups(self, results, executed, prior):
        company, news = self._company(results, prior), [r for r in results if r.tool == "fetch_news"]
        n = sum(len(r.data.get("articles", [])) for r in news if r.ok)
        query = f"{company} stock" if company else None
        already = any(s["tool"] == "fetch_news" and s.get("args", {}).get("query") == query for s in executed)
        if query and any(r.ok for r in news) and n < 5 and not self.news_file and not already:
            return [{"tool": "fetch_news", "args": {"query": query},
                     "why": f"only {n} article(s) found; widening query with company name"}]
        return []

    # ---------- 6. REFLECT & LEARN ----------
    def reflect(self, ctx, opt, results):
        hist, rule_lessons = opt["history"], []
        for r in results:
            if not r.ok and r.tool != "recall_memory":
                rule_lessons.append({"tag": f"tool_failed:{r.tool}",
                                     "text": f"{r.tool} failed ({r.error}); check credentials/inputs or use a fallback."})
        n = len(ctx["digest"]["articles"])
        if n < 5:
            rule_lessons.append({"tag": "news_thin", "text": f"Only {n} usable news articles; widen the query with the company name or load a headline dump."})
        for k, v in hist[0]["scores"].items():
            if v < 4:
                rule_lessons.append({"tag": f"weak:{k}", "text": f"First draft scored {v}/5 on {k}; cover it in the first draft."})
        if len(hist) > 1:
            rule_lessons.append({"tag": "needed_refinement", "text": f"Draft needed {len(hist) - 1} refinement round(s) to pass."})
        final = hist[-1]
        weakest = min(final["scores"], key=final["scores"].get)
        assessment = (f"Final score {final['overall']}/5 ({'passed' if final['passed'] else 'NOT passed'}) after "
                      f"{len(hist) - 1} refinement(s); weakest criterion: {weakest}; {len(ctx['gaps'])} data gap(s).")
        llm_lessons = []
        if self.llm:
            j = parse_json(self.llm(REFLECT_SYSTEM, json.dumps({"history": [{k: h[k] for k in ("iteration", "scores", "feedback")} for h in hist],
                                                                 "gaps": ctx["gaps"]}, default=str)))
            if isinstance(j, dict):
                assessment = j.get("assessment") if isinstance(j.get("assessment"), str) else assessment
                llm_lessons = [x for x in j.get("lessons", []) if isinstance(x, str)][:3]
        return {"assessment": assessment, "rule_lessons": rule_lessons, "llm_lessons": llm_lessons}

    # ---------- RUN ----------
    def run(self):
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S-") + uuid.uuid4().hex[:6]
        prior = self.memory.last_run(self.ticker)
        lessons = self.memory.get_lessons(self.ticker)

        steps, mode = self.plan()
        self.trace.append({"phase": "plan", "mode": mode, "steps": steps})
        executed, results = list(steps), [self._call(s) for s in steps]
        for s in self._followups(results, executed, prior):  # dynamic re-plan
            self.trace.append({"phase": "replan", "step": s})
            executed.append(s)
            results.append(self._call(s))

        sources = {}
        for i, r in enumerate(results, 1):
            sources[f"S{i}"] = {"tool": r.tool, "source": r.source, "ok": r.ok, "error": r.error,
                                "warnings": r.warnings, "fetched_at": r.fetched_at}
        company = self._company(results, prior)

        digest = run_news_chain([r for r in results if r.tool == "fetch_news"], self.ticker, company, self.llm)

        items = [{"kind": "tool", "tool": r.tool, "sid": sid, "result": r}
                 for (sid, r) in zip(sources, results) if r.ok and r.tool in TOOL_ROUTES]
        items += [{"kind": "article", "sid": a["id"], "article": a} for a in digest["articles"]]
        buckets, route_log = Router().route(items)

        gaps = [f"{v['tool']}: {v['error']}" for v in sources.values() if not v["ok"]]
        gaps += [f"{v['tool']}: {w}" for v in sources.values() for w in v["warnings"]]
        if not digest["articles"] and not digest["errors"]:
            gaps.append("news: no relevant articles found")

        price = next((r for r in results if r.tool == "get_price_history"), None)
        fund = next((r for r in results if r.tool == "get_fundamentals"), None)
        metrics = mt.to_memory_metrics(price, fund)
        metrics.update({"company": company, "news_articles": len(digest["articles"])})
        memory_sid = next((k for k, v in sources.items() if v["tool"] == "recall_memory" and v["ok"]), None)

        ctx = {"ticker": self.ticker, "company": company, "run_date": date.today().isoformat(),
               "sources": sources, "digest": digest, "gaps": gaps,
               "valid_ids": list(sources) + [a["id"] for a in digest["articles"]],
               "memory_sid": memory_sid, "lessons": [l["text"] for l in lessons],
               "lesson_tags": [l["tag"] for l in lessons],
               "prior": {"run_id": prior["run_id"]} if prior else None,
               "changes": MemoryStore.diff(prior, metrics)}
        ctx["specialists"] = run_specialists(buckets, ctx)

        opt = optimize(ctx, self.llm, self.max_refinements, self.threshold)
        refl = self.reflect(ctx, opt, results)
        final = opt["history"][-1]
        self.memory.save_run(self.ticker, run_id, metrics, [a["title"] for a in digest["articles"]],
                             final["feedback"], opt["brief"], final["scores"])
        learned = self.memory.update_lessons(self.ticker, refl["rule_lessons"], refl["llm_lessons"])

        os.makedirs(self.log_dir, exist_ok=True)
        path = os.path.join(self.log_dir, f"{self.ticker}_{run_id}.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"run_id": run_id, "ticker": self.ticker, "trace": self.trace, "sources": sources,
                       "route_log": route_log, "news_chain": digest["stage_log"], "evaluation": opt["history"],
                       "reflection": refl, "lessons_saved": learned, "brief": opt["brief"]}, f, indent=2, default=str)
        return {"run_id": run_id, "brief": opt["brief"], "plan_mode": mode, "plan": steps, "route_log": route_log,
                "news_chain": digest["stage_log"], "evaluation": opt["history"], "reflection": refl,
                "lessons": learned, "audit_path": path}


Writing research_agent.py


## Step 12 - Offline tests (fake data, no keys needed): Offline tests will give the agent test the agent's logic independently from external services and test whether our code works correctly without API keys, internet, or external services.

In [ ]:
%%writefile test_pipeline.py
"""Offline tests (no network, no API keys). Run: python test_pipeline.py"""
import json
import os
import sys
import tempfile
import unittest

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import numpy as np
import pandas as pd

import chain
import extra_tools as ex
import market_tools as mt
from memory import MemoryStore
from research_agent import ResearchAgent
from router import Router
from schema import ToolResult


def make_df(n=120):
    idx = pd.bdate_range("2026-01-01", periods=n)
    c = 100 + np.arange(n) * 0.5
    return pd.DataFrame({"Open": c, "High": c + 1, "Low": c - 1, "Close": c, "Volume": np.full(n, 1e6)}, index=idx)


class FakeTicker:
    info = {"marketCap": 3e12, "trailingPE": 34.0, "forwardPE": 30.0, "trailingEps": 6.1,
            "shortName": "Acme Corp", "sector": "Tech", "currency": "USD"}
    calendar = {}

    def __init__(self, n=120):
        self.n = n

    def history(self, **kw):
        return make_df(self.n)


ARTICLES = [
    {"title": "Acme beats quarterly earnings estimates, raises guidance", "body": "Revenue rose 8% to $90 billion.", "source": "Wire", "published_at": "2026-09-28"},
    {"title": "Acme faces antitrust lawsuit from regulators", "body": "The DOJ probe concerns app store rules.", "source": "Wire2", "published_at": "2026-09-27"},
    {"title": "Analyst upgrades Acme, lifts price target", "body": "Acme rating raised to outperform.", "source": "Bank", "published_at": "2026-09-26"},
    {"title": "Acme unveils new product line", "body": "Acme launch event announces features.", "source": "Blog", "published_at": "2026-09-25"},
    {"title": "Acme unveils new product line", "body": "duplicate headline", "source": "Blog", "published_at": "2026-09-25"},
    {"title": "Fed signals rate cut as inflation cools", "body": "Acme shares gain on macro optimism.", "source": "News", "published_at": "2026-09-24"},
    {"title": "Unrelated company news", "body": "Nothing to see", "source": "X", "published_at": "2026-09-24"},
]


def tools(n=120, articles=ARTICLES):
    return {
        "get_price_history": {"fn": lambda t, period="6mo": mt.get_price_history(t, ticker_factory=lambda s: FakeTicker(n)),
                              "desc": "prices", "args": ["period"]},
        "get_fundamentals": {"fn": lambda t: mt.get_fundamentals(t, ticker_factory=lambda s: FakeTicker()),
                             "desc": "fundamentals", "args": []},
        "fetch_news": {"fn": lambda t, query=None: ToolResult.success(
            "fetch_news", {"query": query or t, "count": len(articles), "articles": articles}, "stub"),
            "desc": "news", "args": ["query"]},
    }


class StubLLM:
    """Returns valid JSON for chain/plan/judge/reflect; returns '' for writer/refine so template fallback is used."""
    def __init__(self):
        self.seen = []

    def __call__(self, system, user):
        self.seen.append(system[:30])
        if "planning module" in system:
            return '```json\n{"steps":[{"tool":"get_price_history","args":{"period":"3mo","bogus":1},"why":"x"},{"tool":"nonexistent","args":{}}]}\n```'
        if "classify financial news" in system:
            return json.dumps([{"id": a["id"], "category": "earnings", "sentiment": "positive"} for a in json.loads(user)][:2])
        if "extract facts" in system:
            return json.dumps([{"id": a["id"], "facts": ["fact for " + a["id"]], "numbers": ["8%"]} for a in json.loads(user)])
        if "news summaries" in system:
            return "Earnings tone was constructive [N1] with legal overhang [N99]."  # invalid id -> gate must reject
        if "strict reviewer" in system:
            return '{"scores":{"grounding":5,"completeness":5,"balance":5,"transparency":5,"memory_use":5},"feedback":[]}'
        if "reviewing your own run" in system:
            return '{"assessment":"Good run.","lessons":["Check news volume early."]}'
        return ""


class ChainTests(unittest.TestCase):
    def test_rules_only_chain(self):
        r = ToolResult.success("fetch_news", {"articles": ARTICLES}, "stub")
        d = chain.run_news_chain([r], "ACME", "Acme Corp")
        ids = {a["id"] for a in d["articles"]}
        self.assertEqual(len(ids), 5)  # dup and unrelated removed
        cats = {a["title"]: a["category"] for a in d["articles"]}
        self.assertEqual(cats["Acme beats quarterly earnings estimates, raises guidance"], "earnings")
        self.assertEqual(cats["Acme faces antitrust lawsuit from regulators"], "legal_regulatory")
        self.assertEqual(cats["Analyst upgrades Acme, lifts price target"], "analyst_rating")
        self.assertEqual([s["stage"] for s in d["stage_log"]], ["ingest", "preprocess", "classify", "extract", "summarize"])
        self.assertTrue(all("[N" in d["summary"] for _ in [0]))

    def test_llm_gates_fall_back(self):
        r = ToolResult.success("fetch_news", {"articles": ARTICLES}, "stub")
        d = chain.run_news_chain([r], "ACME", "Acme Corp", StubLLM())
        log = {s["stage"]: s for s in d["stage_log"]}
        self.assertEqual(log["classify"]["by_llm"], 2)          # stub labeled only 2
        self.assertEqual(log["classify"]["by_rules"], 3)        # rest fell back
        self.assertEqual(log["summarize"]["source"], "rules")   # invalid [N99] citation rejected

    def test_no_news(self):
        d = chain.run_news_chain([ToolResult.failure("fetch_news", "401")], "ACME")
        self.assertEqual(d["articles"], [])
        self.assertIn("No usable", d["summary"])


class RouterTests(unittest.TestCase):
    def test_routes(self):
        arts = [{"id": "N1", "category": "earnings"}, {"id": "N2", "category": "legal_regulatory"},
                {"id": "N3", "category": "analyst_rating"}]
        items = [{"kind": "article", "sid": a["id"], "article": a} for a in arts]
        items.append({"kind": "tool", "tool": "get_price_history", "sid": "S2", "result": None})
        items.append({"kind": "tool", "tool": "get_fundamentals", "sid": "S3", "result": None})
        b, log = Router().route(items)
        self.assertEqual([i["sid"] for i in b["earnings"]], ["N1", "N3", "S3"])
        self.assertEqual([i["sid"] for i in b["news"]], ["N2"])
        self.assertEqual([i["sid"] for i in b["market"]], ["S2"])
        self.assertEqual(len(log), 5)


class ExtraToolTests(unittest.TestCase):
    def test_fred(self):
        fake = lambda url: {"observations": [{"date": "2026-08-01", "value": "."}, {"date": "2026-07-01", "value": "4.33"}]}
        r = ex.get_macro(("FEDFUNDS",), http_get=fake)
        self.assertTrue(r.ok)
        self.assertEqual(r.data["series"][0]["value"], 4.33)
        self.assertFalse(ex.get_macro(api_key=None).ok if not os.getenv("FRED_API_KEY") else False)

    def test_edgar(self):
        def fake(url):
            if "company_tickers" in url:
                return {"0": {"cik_str": 320193, "ticker": "ACME", "title": "Acme"}}
            return {"filings": {"recent": {"form": ["8-K", "4", "10-Q"], "filingDate": ["2026-09-01", "2026-08-30", "2026-08-01"],
                                           "accessionNumber": ["1-1-1", "2-2-2", "3-3-3"], "primaryDocument": ["a.htm", "b.htm", "c.htm"]}}}
        r = ex.get_edgar_filings("acme", http_get=fake)
        self.assertEqual([f["form"] for f in r.data["filings"]], ["8-K", "10-Q"])
        self.assertFalse(ex.get_edgar_filings("ZZZ", http_get=fake).ok)


class AgentTests(unittest.TestCase):
    def test_learning_across_runs_offline(self):
        with tempfile.TemporaryDirectory() as d:
            mem = MemoryStore(os.path.join(d, "m.json"))
            first = ResearchAgent("ACME", memory=mem, tools=tools(120), log_dir=d).run()
            h1 = first["evaluation"]
            self.assertGreaterEqual(len(h1), 2)                  # lean draft needed refinement
            self.assertLess(h1[0]["overall"], h1[-1]["overall"])  # optimizer improved the score
            self.assertTrue(h1[-1]["passed"])
            tags = {l["tag"] for l in first["lessons"]}
            self.assertIn("weak:balance", tags)

            second = ResearchAgent("ACME", memory=mem, tools=tools(130), log_dir=d).run()
            self.assertEqual(len(second["evaluation"]), 1)       # LEARNED: first draft already passes
            self.assertTrue(second["evaluation"][0]["passed"])
            self.assertIn("last_close", second["brief"])          # memory diff reported
            self.assertNotIn("No prior run", second["brief"].split("## Changes since last run")[1].split("##")[0])
            self.assertTrue(os.path.exists(second["audit_path"]))

    def test_replan_widens_thin_news_and_disclosure(self):
        with tempfile.TemporaryDirectory() as d:
            few = ARTICLES[:2]
            calls = []
            t = tools(articles=few)
            base = t["fetch_news"]["fn"]
            t["fetch_news"]["fn"] = lambda tk, query=None: (calls.append(query), base(tk, query=query))[1]
            out = ResearchAgent("ACME", memory=MemoryStore(os.path.join(d, "m.json")), tools=t, log_dir=d).run()
            self.assertEqual(calls, [None, "Acme Corp stock"])   # second, widened call
            self.assertIn("news_thin", {l["tag"] for l in out["lessons"]})

    def test_tool_failure_is_disclosed_and_learned(self):
        with tempfile.TemporaryDirectory() as d:
            t = tools()
            t["fetch_news"]["fn"] = lambda tk, query=None: ToolResult.failure("fetch_news", "HTTPError: 401", "NewsAPI")
            out = ResearchAgent("ACME", memory=MemoryStore(os.path.join(d, "m.json")), tools=t, log_dir=d).run()
            self.assertIn("HTTPError: 401", out["brief"].split("## Data gaps")[1])
            self.assertIn("tool_failed:fetch_news", {l["tag"] for l in out["lessons"]})

    def test_llm_path_with_stub(self):
        with tempfile.TemporaryDirectory() as d:
            llm = StubLLM()
            out = ResearchAgent("ACME", llm=llm, memory=MemoryStore(os.path.join(d, "m.json")), tools=tools(), log_dir=d).run()
            self.assertEqual(out["plan_mode"], "llm")
            names = [s["tool"] for s in out["plan"]]
            self.assertNotIn("nonexistent", names)
            self.assertTrue({"get_fundamentals", "fetch_news", "recall_memory"} <= set(names))  # safeguards
            self.assertEqual(out["reflection"]["assessment"], "Good run.")
            self.assertIn("Check news volume early.", [l["text"] for l in out["lessons"]])
            self.assertTrue(out["evaluation"][-1]["passed"])


if __name__ == "__main__":
    unittest.main(verbosity=1)


Writing test_pipeline.py


## Step 12b - Run the tests: This section will test whether the data is valid or fake market data, Missing columns, Empty data, Invalid ticker, API failure,  Missing fundamentals, and report whether is ToolResult.success() or ToolResult.failure()


In [ ]:
!python test_pipeline.py

..........
----------------------------------------------------------------------
Ran 10 tests in 0.166s

OK


## Step 13 - Run the agent (first run)
Uses Gemini if GEMINI_API_KEY is set, otherwise the rule-based fallbacks. Change TICKER freely.  

In [ ]:
import importlib, os, json
import research_agent, chain, router, evaluator
for m in (chain, router, evaluator, research_agent): importlib.reload(m)
from research_agent import ResearchAgent
from llm import make_llm
from memory import MemoryStore

TICKER = 'AAPL'
NEWS_FILE = None            # e.g. 'headlines.json'
llm = make_llm('gemini')   # returns None if GEMINI_API_KEY is not set
print(type(llm).__name__ + ' (' + llm.model + ')' if llm else 'Rule-based mode (no GEMINI_API_KEY)')

agent = ResearchAgent(TICKER, llm=llm, memory=MemoryStore('memory/store.json'), news_file=NEWS_FILE)
out = agent.run()
print(out['brief'])

GeminiLLM (gemini-2.5-flash)
# AAPL Analyst Note (2026-09-29)

## Summary
- AAPL closed at 329.4, 33.8% over 127 sessions, trading between of its 20/50-day averages [S2]
- Valued at 37.7x trailing earnings (34.4x forward); next earnings 2026-10-29 [S3]
- Non-earnings news tone is positive (3 positive vs 1 negative of 12) [N1][N2][N3]

## Market
- Last close 329.4 on 2026-09-29 [S2]
- Return 33.8% over 127 sessions; last month 3.0% [S2]
- 20-day average 330.96 and 50-day average 321.90; price is between of them [S2]
- Sits at 84% of its period range (245.0726 to 345.34) [S2]
- Annualized volatility 27.7% [S2]
- Macro: FEDFUNDS = 3.63 (2026-08-01) [S5]
- Macro: DGS10 = 5.24 (2026-09-28) [S5]
- Macro: CPIAUCSL = 334.131 (2026-08-01) [S5]

## Earnings & Fundamentals
- Apple Inc. (Technology), market cap $4.81T [S3]
- Trailing P/E 37.7, forward P/E 34.4, trailing EPS 8.73 [S3]
- Forward P/E below trailing implies expected earnings growth [S3]
- Next earnings 2026-10-29 (in 30 days) [S3]
- R


Next Step: Team B and C

